# NB02 · Semana 4 — Extracción de la API hacia la capa bronze

**Ingeniería de Datos — LCD · UCA Rosario · 2026**

Fuente del proyecto: **VATSIM** (`https://data.vatsim.net/v3/vatsim-data.json`), red de tráfico aéreo simulado. Gratis, sin API key, sin cuota.

## Objetivo

Construir el primer tramo del pipeline: **extraer** el snapshot de pilotos conectados a VATSIM y depositarlo en la capa **bronze** del lake como tabla Delta, aplicando dos patrones clave:

- **Carga incremental**: traer solo lo nuevo desde la última corrida (*watermark* sobre `snapshot_ts`).
- **Idempotencia**: reejecutar el pipeline no duplica datos ni rompe nada.

> **Nota sobre la fuente.** VATSIM no ofrece histórico ni filtro por fecha: cada llamada es una **foto del momento** (se actualiza cada ~15 s). El histórico se **construye acumulando snapshots** en bronze, una corrida = una foto, cada una con su `snapshot_ts`. Es un caso legítimo de ingesta incremental que genera la serie temporal.

> Este notebook es la base de la **Entrega 1 (E1)** del proyecto.

## 1. Configuración del acceso al lake

El mismo notebook corre en el Docker local y en el servidor de la cátedra: `entorno.py` (el de NB01) resuelve el bucket y las credenciales según el entorno.

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, STORAGE_OPTIONS, info
info()

# VATSIM va en su propia tabla dentro de bronze del bucket del grupo.
BRONZE = f's3://{BUCKET}/bronze/vatsim'
BRONZE_TEST = f's3://{BUCKET}/bronze/_test_vatsim'  # pruebas: no ensucia bronze real
print('Destino bronze:', BRONZE)

## 2. Leer el *watermark*

El watermark es el máximo `snapshot_ts` ya cargado en bronze. Si la tabla no existe todavía, devolvemos `None` (primera corrida).

Solo capturamos `TableNotFoundError`: cualquier otro error (MinIO caído, 403 por permisos) **debe fallar a la vista**, no hacerse pasar por una primera corrida.

In [ ]:
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError


def leer_watermark(destino: str = BRONZE) -> str | None:
    """Devuelve el máximo 'snapshot_ts' ya presente en bronze.

    Args:
        destino: URI de la tabla Delta de bronze.

    Returns:
        Timestamp máximo ya ingerido, o None si la tabla aún no existe
        o no tiene snapshots (primera corrida).
    """
    try:
        dt = DeltaTable(destino, storage_options=STORAGE_OPTIONS)
    except TableNotFoundError:
        return None
    ts = dt.to_pandas(columns=['snapshot_ts'])['snapshot_ts'].dropna()
    return None if ts.empty else str(ts.max())


print('watermark actual:', leer_watermark())

## 3. Extraer de la API de VATSIM

Un solo endpoint, sin autenticación ni paginación: una llamada devuelve todos los pilotos conectados. El campo `general.update_timestamp` es el sello de tiempo del snapshot y hace de watermark.

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://data.vatsim.net/v3/vatsim-data.json"


def extraer_de_api(desde: str | None = None) -> pd.DataFrame:
    """Extrae el snapshot actual de pilotos conectados a VATSIM.

    VATSIM no tiene histórico ni filtro por fecha: cada llamada es una foto
    del momento. El histórico se construye acumulando snapshots en bronze,
    uno por corrida, cada uno identificado por 'snapshot_ts'.

    Args:
        desde: Watermark de la última corrida (update_timestamp ya ingerido).
            Si el snapshot actual no es más nuevo, se devuelve vacío para no
            reingerir la misma foto (idempotencia por watermark).

    Returns:
        DataFrame de pilotos, todo como string (bronze crudo; se tipa en silver),
        con la columna 'snapshot_ts'. El objeto anidado flight_plan se aplana
        a columnas flight_plan_* (queda nulo si el piloto no cargó plan).

    Raises:
        RuntimeError: Si VATSIM no responde correctamente.
    """
    try:
        r = requests.get(BASE_URL, timeout=30)
        r.raise_for_status()
    except requests.RequestException as exc:
        status = getattr(exc.response, "status_code", None)
        raise RuntimeError(f"Falló VATSIM ({type(exc).__name__}, HTTP {status})") from None

    data = r.json()
    snapshot_ts = data["general"]["update_timestamp"]
    if desde and snapshot_ts <= desde:
        print(f"Sin snapshot nuevo (watermark={desde}).")
        return pd.DataFrame()

    df = pd.json_normalize(data["pilots"], sep="_")
    df["snapshot_ts"] = snapshot_ts
    print(f"Filas extraídas: {len(df)} (snapshot {snapshot_ts})")
    return df.astype("string")


nuevos = extraer_de_api(desde=leer_watermark())
print(nuevos.shape); nuevos.head()

### Alternativa sin red: snapshot sintético

Para probar la ingesta y la idempotencia **sin depender del reloj** (VATSIM refresca cada ~15 s, así que dos corridas reales seguidas caen en fotos distintas). El `snapshot_ts` acá es **fijo**, así la segunda corrida siempre da 0. Se escribe en `BRONZE_TEST`.

In [ ]:
def extraer_mock(desde: str | None = None) -> pd.DataFrame:
    """Snapshot VATSIM sintético y FIJO, para verificar idempotencia."""
    df = pd.DataFrame({
        "cid": ["1544123", "1472003", "1986825", "1962126", "1428384"],
        "callsign": ["DLH112", "UAE248", "SIA34", "KLM836", "QFA93"],
        "latitude": ["25.26", "25.24", "36.82", "52.31", "27.05"],
        "longitude": ["51.61", "55.35", "-121.85", "4.76", "-128.88"],
        "altitude": ["22", "27", "41052", "4", "39053"],
        "snapshot_ts": ["2026-09-25T22:00:00.0000000Z"] * 5,  # fijo a propósito
    }).astype("string")
    if desde and "2026-09-25T22:00:00.0000000Z" <= desde:
        return pd.DataFrame()
    return df


print("Snapshot sintético disponible para pruebas sin depender del reloj.")

## 4. Escribir en bronze (Delta, append idempotente)

Primera corrida: crea la tabla. Corridas siguientes: agrega solo snapshots nuevos.

**Clave natural:** `cid + snapshot_ts` — un piloto (identificado por su `cid`, único en VATSIM) en una foto dada. Dos barreras de idempotencia: el watermark evita traer la misma foto, y el anti-join por la clave evita duplicar aunque el snapshot se repita.

In [ ]:
import uuid
from datetime import datetime, timezone

CLAVE = ["cid", "snapshot_ts"]


def ingerir(extractor, destino: str = BRONZE) -> int:
    """Ingiere en bronze solo las filas nuevas (incremental e idempotente).

    Args:
        extractor: Función que recibe 'desde' y devuelve un DataFrame de pilotos.
        destino: URI de la tabla Delta de bronze.

    Returns:
        Cantidad de filas escritas (0 si no había novedades).
    """
    wm = leer_watermark(destino)
    nuevos = extractor(desde=wm)
    if nuevos is None or nuevos.empty:
        print(f"Sin novedades (watermark={wm}).")
        return 0

    nuevos = nuevos.drop_duplicates(CLAVE)

    # Segunda barrera de idempotencia: descartar claves que ya existen en bronze.
    try:
        existentes = DeltaTable(destino, storage_options=STORAGE_OPTIONS).to_pandas(columns=CLAVE)
        claves_viejas = set(map(tuple, existentes.values))
        nuevos = nuevos[~nuevos[CLAVE].apply(tuple, axis=1).isin(claves_viejas)]
    except TableNotFoundError:
        pass  # primera corrida

    if nuevos.empty:
        print(f"Todo lo extraído ya estaba en bronze (watermark={wm}).")
        return 0

    nuevos = nuevos.assign(
        _ingested_at=datetime.now(timezone.utc).isoformat(),
        _run_id=uuid.uuid4().hex,
    )
    write_deltalake(destino, nuevos, mode="append", storage_options=STORAGE_OPTIONS)
    print(f"Ingeridas {len(nuevos)} filas (watermark previo={wm}).")
    return len(nuevos)


n = ingerir(extraer_de_api)
print("Filas ingeridas:", n)

## 5. Verificar idempotencia

Con la fuente real no se puede demostrar de forma estable, porque VATSIM cambia el snapshot cada ~15 s: dos corridas seguidas caen en fotos distintas (y eso, lejos de ser un error, es cómo se acumula el histórico). Para demostrar la idempotencia usamos el snapshot **sintético y fijo** sobre `BRONZE_TEST`: la segunda corrida tiene que dar 0.

In [ ]:
n1 = ingerir(extraer_mock, destino=BRONZE_TEST)
n2 = ingerir(extraer_mock, destino=BRONZE_TEST)
print(f"Corrida 1: {n1} filas | Corrida 2: {n2} filas")

df_test = DeltaTable(BRONZE_TEST, storage_options=STORAGE_OPTIONS).to_pandas()
dup = int(df_test.duplicated(CLAVE).sum())
print("Filas en bronze de prueba:", len(df_test), "| duplicadas:", dup)

assert n2 == 0, "La segunda corrida no debería agregar filas"
assert dup == 0, "Hay claves duplicadas en bronze"
print("Idempotencia verificada.")

## 6. Manejo de errores (documentación para la rúbrica)

La rúbrica de la E1 pide documentar el manejo de errores. Casos previstos:

| Situación | Cómo se maneja |
|---|---|
| VATSIM no responde / error HTTP | `extraer_de_api` levanta `RuntimeError` con el código, **antes** de tocar bronze; el watermark no avanza. |
| Snapshot sin cambios (misma foto) | El watermark detecta `snapshot_ts <= desde` y devuelve vacío: 0 filas, sin duplicar. |
| Snapshot repetido que igual llega a `ingerir` | El anti-join por `cid + snapshot_ts` descarta lo ya presente (segunda barrera). |
| Tabla inexistente (primera corrida) | Se captura solo `TableNotFoundError` → `None`; cualquier otro error (403, MinIO caído) **sí** se propaga. |
| Falla al escribir en bronze | El watermark se lee al inicio de la próxima corrida desde la tabla real, así que un fallo de escritura no lo deja desincronizado. |

**Trazabilidad:** cada fila lleva `_ingested_at` y `_run_id`, para linaje en bronze (append-only).

## Entrega 1 (E1)

Lo que se evalúa: ingesta idempotente hacia bronze con carga incremental y manejo de errores documentado.

**Pendiente:** confirmar con el profesor el cambio de fuente a VATSIM (tráfico simulado, sin histórico propio), porque afecta la ficha de dataset.